# A3-L1 practical: Prove a MERGE is idempotent

Platform: Databricks Free Edition.

Run this in Databricks Free Edition on serverless compute. It creates a schema called `learning_a3` in your current catalog, loads the same data twice, and shows that the row count does not change. It then repeats a key and shows the error.


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F

CATALOG = spark.catalog.currentCatalog()
SCHEMA = f"{CATALOG}.learning_a3"
TABLE = f"{SCHEMA}.nat_respondents"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
spark.sql(
    f"CREATE TABLE {TABLE} (respondent_id STRING, wave_id STRING, "
    "final_weight DOUBLE, delivery_version INT)"
)

In [ ]:
def latest_per_key(df: DataFrame, keys: list[str], order_col: str) -> DataFrame:
    """Keep the row with the highest order_col for each key."""
    window = Window.partitionBy(*keys).orderBy(F.col(order_col).desc())
    return (
        df.withColumn("_rank", F.row_number().over(window))
        .filter(F.col("_rank") == 1)
        .drop("_rank")
    )


def upsert_silver(
    spark: SparkSession, source: DataFrame, target_table: str, keys: list[str], order_col: str
) -> None:
    """MERGE the newest source row per key into a silver table."""
    condition = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    (
        DeltaTable.forName(spark, target_table).alias("t")
        .merge(latest_per_key(source, keys, order_col).alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

Load the same three rows twice. The count stays at 3.

In [ ]:
source = spark.createDataFrame(
    [("R1", "W1", 1.2, 1), ("R2", "W1", 0.8, 1), ("R3", "W1", 1.0, 1)],
    "respondent_id STRING, wave_id STRING, final_weight DOUBLE, delivery_version INT",
)
KEYS = ["respondent_id", "wave_id"]
upsert_silver(spark, source, TABLE, KEYS, "delivery_version")
upsert_silver(spark, source, TABLE, KEYS, "delivery_version")
print(spark.table(TABLE).count())  # 3

Now a redelivery repeats R1. Merge it without deduplicating and read the error. Then run the deduplicating version and check that R1 carries version 2.

In [ ]:
redelivery = spark.createDataFrame(
    [("R1", "W1", 1.5, 2), ("R1", "W1", 1.4, 2)],
    "respondent_id STRING, wave_id STRING, final_weight DOUBLE, delivery_version INT",
)
try:
    (
        DeltaTable.forName(spark, TABLE).alias("t")
        .merge(redelivery.alias("s"), "t.respondent_id = s.respondent_id AND t.wave_id = s.wave_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
except Exception as err:  # noqa: BLE001 - show the error class for learning
    print(type(err).__name__, str(err)[:300])

In [ ]:
upsert_silver(spark, redelivery, TABLE, KEYS, "delivery_version")
display(spark.table(TABLE).orderBy("respondent_id"))